
####Objetivo

Registrar el modelo de clasificación seleccionado en MLflow Unity Catalog y preparar su despliegue como un servicio REST en Databricks para la realización de predicciones sobre nuevos registros.

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

In [0]:
# Modelo ganador seleccionado en el Notebook 04_6

run_id_ganador = "f17c2e3c69f74ddc8bda80eb77f14b00"

modelo_uri = f"runs:/{run_id_ganador}/modelo"

print(modelo_uri)

In [0]:
modelo_ganador = mlflow.sklearn.load_model(modelo_uri)

print("Modelo cargado correctamente.")
print(type(modelo_ganador))

In [0]:
# Registrar el modelo en MLflow

nombre_modelo = (
    "ml_proyecto_7405607705157039.default.prediccion_nivel_peligro"
)

client = MlflowClient()

# Buscar versiones existentes del modelo
versiones = client.search_model_versions(
    f"name='{nombre_modelo}'"
)

# Verificar si el run_id ya está registrado
modelo_ya_registrado = any(
    version.run_id == run_id_ganador
    for version in versiones
)

if modelo_ya_registrado:
    print("El modelo ya se encuentra registrado. No se crea una nueva versión.")
else:
    modelo_registrado = mlflow.register_model(
        model_uri=modelo_uri,
        name=nombre_modelo
    )
    print(f"Modelo registrado correctamente. Versión: {modelo_registrado.version}")

# Mostrar las versiones registradas

versiones = client.search_model_versions(
    f"name='{nombre_modelo}'"
)

print("\nVersiones registradas:")

for version in versiones:
    print(
        f"Versión: {version.version} | "
        f"Run ID: {version.run_id} | "
        f"Estado: {version.status}"
    )

#### Despliegue del modelo mediante Model Serving de Databricks

Una vez seleccionado el modelo de mejor desempeño (Regresión Logística, versión 3), este fue registrado exitosamente en el MLflow Model Registry bajo el nombre prediccion_nivel_peligro. Posteriormente, se inició el proceso de creación de un Serving Endpoint para exponer el modelo como un servicio REST.

Durante este proceso, Databricks mostró el siguiente mensaje:

Model serving is not available for trial workspaces.

Este mensaje indica que la funcionalidad Model Serving no está disponible en los espacios de trabajo de tipo Trial de Databricks. En consecuencia, aunque el modelo se encontraba correctamente registrado y listo para su despliegue, la creación del endpoint no pudo completarse debido a una restricción de licenciamiento de la plataforma y no a un error del modelo ni del procedimiento de implementación.

Por lo tanto, el flujo de despliegue quedó validado hasta la etapa de configuración del endpoint, siendo necesario disponer de un workspace de Databricks con una suscripción de pago para habilitar el servicio de inferencia REST administrado por Databricks.